# Code-Switching Model Training
**Rulează cu GPU:** Runtime → Change runtime type → T4 GPU

La final se descarcă automat `cs_model.zip` — dezarhivează și pune folderul în proiect.

In [1]:
# ── CELL 1 — Instalare pachete ──────────────────────────────────────
!pip install -q transformers datasets seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# ── CELL 2 — Imports ────────────────────────────────────────────────
import os
import numpy as np
import requests
import torch
from pathlib import Path
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import f1_score, classification_report

print('GPU disponibil:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

GPU disponibil: True
GPU: Tesla T4


In [3]:
# ── CELL 3 — Config ─────────────────────────────────────────────────
MODEL_NAME  = 'xlm-roberta-base'
SAVE_PATH   = '/content/cs_model'
EPOCHS      = 3
BATCH_SIZE  = 32

LABEL_NAMES = ['lang1', 'lang2', 'mixed', 'ne', 'fw', 'unk', 'other']
label2id    = {n: i for i, n in enumerate(LABEL_NAMES)}
id2label    = {i: n for i, n in enumerate(LABEL_NAMES)}

print('Config OK')
print('Labels:', LABEL_NAMES)

Config OK
Labels: ['lang1', 'lang2', 'mixed', 'ne', 'fw', 'unk', 'other']


In [4]:
# CELL 4 — Date sintetice Română-Engleză (romgleză)
L1 = 'lang1'  # Română
L2 = 'lang2'  # Engleză
OT = 'other'

RAW_EXAMPLES = [
    # facultate
    (['Deadline-ul', 'e', 'maine', 'si', 'n-am', 'facut', 'nimic'],         [L2, L1, L1, L1, L1, L1, L1]),
    (['Am', 'picat', 'examenul', 'si', 'sunt', 'in', 'full', 'panic'],       [L1, L1, L1, L1, L1, L1, L2, L2]),
    (['Bro', 'seriously', 'nu', 'mai', 'pot', 'cu', 'cursurile', 'astea'],   [L2, L2, L1, L1, L1, L1, L1, L1]),
    (['Trebuie', 'sa', 'submit', 'proiectul', 'pana', 'la', 'ora', '5'],     [L1, L1, L2, L1, L1, L1, L1, OT]),
    (['Nu', 'am', 'fost', 'la', 'curs', 'azi', 'am', 'overslept'],           [L1, L1, L1, L1, L1, L1, L1, L2]),
    (['Grupa', 'mea', 'e', 'full', 'de', 'free-rideri'],                     [L1, L1, L1, L2, L1, L2]),
    (['Profesorul', 'a', 'dat', 'assignment', 'pentru', 'vineri'],            [L1, L1, L1, L2, L1, L1]),
    (['Am', 'luat', 'feedback', 'bun', 'la', 'proiect'],                     [L1, L1, L2, L1, L1, L1]),
    (['Trebuie', 'sa', 'fac', 'research', 'pentru', 'lucrare'],              [L1, L1, L1, L2, L1, L1]),
    (['Nu', 'stiu', 'cum', 'sa', 'implement', 'asta'],                       [L1, L1, L1, L1, L2, L1]),
    # gaming
    (['Am', 'dat', 'un', 'carry', 'azi', 'toata', 'echipa', 'era', 'trash'], [L1, L1, L1, L2, L1, L1, L1, L1, L2]),
    (['Bro', 'ce', 'feed', 'ai', 'dat', 'azi'],                              [L2, L1, L2, L1, L1, L1]),
    (['Am', 'luat', 'ban', 'pe', 'cont', 'pentru', 'nimic'],                 [L1, L1, L2, L1, L1, L1, L1]),
    (['Rank-ul', 'meu', 'a', 'scazut', 'dupa', 'update'],                    [L2, L1, L1, L1, L1, L2]),
    (['Hai', 'sa', 'facem', 'un', 'game', 'diseara'],                        [L1, L1, L1, L1, L2, L1]),
    (['Mi-a', 'placut', 'gameplay-ul', 'dar', 'story-ul', 'e', 'slab'],      [L1, L1, L2, L1, L2, L1, L1]),
    (['Am', 'stat', 'toata', 'noaptea', 'online', 'si', 'sunt', 'mort'],     [L1, L1, L1, L1, L2, L1, L1, L1]),
    (['Server-ul', 'a', 'crapat', 'la', 'mijlocul', 'jocului'],              [L2, L1, L1, L1, L1, L1]),
    # munca
    (['Meeting-ul', 'de', 'azi', 'a', 'fost', 'waste', 'of', 'time'],        [L2, L1, L1, L1, L1, L2, L2, L2]),
    (['Seful', 'vrea', 'report', 'pana', 'maine', 'dimineata'],              [L1, L1, L2, L1, L1, L1]),
    (['Fac', 'internship', 'la', 'o', 'firma', 'de', 'software'],            [L1, L2, L1, L1, L1, L1, L2]),
    (['Am', 'trimis', 'email-ul', 'dar', 'n-am', 'primit', 'reply'],         [L1, L1, L2, L1, L1, L1, L2]),
    (['Trebuie', 'sa', 'fac', 'prezentarea', 'pentru', 'client'],            [L1, L1, L1, L1, L1, L2]),
    (['Am', 'luat', 'un', 'job', 'part-time', 'la', 'cafenea'],              [L1, L1, L1, L2, L2, L1, L1]),
    (['Proiectul', 'e', 'on', 'track', 'deocamdata'],                        [L1, L1, L2, L2, L1]),
    # social media
    (['Ce', 'faci', 'bro', 'how', 'are', 'you'],                             [L1, L1, L2, L2, L2, L2]),
    (['Sunt', 'asa', 'tired', 'dupa', 'ieri'],                               [L1, L1, L2, L1, L1]),
    (['Bro', 'nu', 'mai', 'pot', 'seriously'],                               [L2, L1, L1, L1, L2]),
    (['Ce', 'outfit', 'misto', 'ai', 'azi'],                                 [L1, L2, L1, L1, L1]),
    (['Am', 'postat', 'o', 'poza', 'si', 'am', 'luat', 'hate'],              [L1, L1, L1, L1, L1, L1, L1, L2]),
    (['Urmareste-ma', 'pe', 'instagram', 'please'],                          [L1, L1, L2, L2]),
    (['Story-ul', 'tau', 'e', 'super', 'funny'],                             [L2, L1, L1, L2, L2]),
    (['Am', 'vazut', 'un', 'video', 'viral', 'si', 'am', 'ras', 'mult'],     [L1, L1, L1, L2, L2, L1, L1, L1, L1]),
    # tehnologie
    (['Am', 'dat', 'update', 'la', 'laptop', 'si', 'acum', 'crapa', 'tot'], [L1, L1, L2, L1, L2, L1, L1, L1, L1]),
    (['Telefonul', 'meu', 'are', 'nevoie', 'de', 'charge'],                  [L1, L1, L1, L1, L1, L2]),
    (['Am', 'instalat', 'un', 'app', 'nou', 'si', 'e', 'super', 'tare'],    [L1, L1, L1, L2, L1, L1, L1, L2, L1]),
    (['Wifi-ul', 'de', 'la', 'facultate', 'e', 'trash'],                     [L2, L1, L1, L1, L1, L2]),
    (['Am', 'facut', 'backup', 'la', 'toate', 'fisierele'],                  [L1, L1, L2, L1, L1, L1]),
    (['Browserul', 'meu', 'are', 'prea', 'multe', 'tab-uri', 'deschise'],   [L2, L1, L1, L1, L1, L2, L1]),
    # iesit in oras
    (['Mergem', 'out', 'diseara', 'you', 'coming'],                          [L1, L2, L1, L2, L2]),
    (['Localul', 'ala', 'e', 'super', 'nice', 'te', 'duc', 'acolo'],        [L1, L1, L1, L2, L2, L1, L1, L1]),
    (['Am', 'stat', 'la', 'coada', 'o', 'ora', 'pentru', 'brunch'],         [L1, L1, L1, L1, L1, L1, L1, L2]),
    (['Mancarea', 'a', 'fost', 'amazing', 'dar', 'scumpa'],                  [L1, L1, L1, L2, L1, L1]),
    (['Hai', 'sa', 'mergem', 'la', 'un', 'coffee', 'dupa', 'cursuri'],      [L1, L1, L1, L1, L1, L2, L1, L1]),
    # oboseala
    (['Merg', 'la', 'gym', 'dupa', 'work', 'desi', 'sunt', 'mort'],         [L1, L1, L2, L1, L2, L1, L1, L1]),
    (['Am', 'dormit', '4', 'ore', 'si', 'trebuie', 'sa', 'fiu', 'okay'],    [L1, L1, OT, L1, L1, L1, L1, L1, L2]),
    (['Shift-ul', 'a', 'fost', 'lung', 'si', 'sunt', 'asa', 'tired'],       [L2, L1, L1, L1, L1, L1, L1, L2]),
    (['Nu', 'am', 'energie', 'deloc', 'azi', 'I', 'give', 'up'],            [L1, L1, L1, L1, L1, L2, L2, L2]),
    (['Trebuie', 'sa', 'ma', 'trezesc', 'la', '6', 'kill', 'me'],           [L1, L1, L1, L1, L1, OT, L2, L2]),
    (['Azi', 'am', 'avut', 'o', 'zi', 'full', 'on', 'grea'],                [L1, L1, L1, L1, L1, L2, L2, L1]),
]

# Convertim etichetele la id-uri
EXAMPLES = [
    (words, [label2id[l] for l in labels])
    for words, labels in RAW_EXAMPLES
]

import random
random.seed(42)
EXAMPLES = EXAMPLES * 60
random.shuffle(EXAMPLES)

split = int(len(EXAMPLES) * 0.85)
train_ex = EXAMPLES[:split]
val_ex   = EXAMPLES[split:]

raw_dataset = DatasetDict({
    'train': Dataset.from_dict({
        'words': [e[0] for e in train_ex],
        'lid':   [e[1] for e in train_ex],
    }),
    'validation': Dataset.from_dict({
        'words': [e[0] for e in val_ex],
        'lid':   [e[1] for e in val_ex],
    }),
})

print(f'Train:  {len(raw_dataset["train"])} propozitii')
print(f'Val:    {len(raw_dataset["validation"])} propozitii')
print(f'Exemplu: {RAW_EXAMPLES[0]}')
print('raw_dataset OK!')

Train:  2550 propozitii
Val:    450 propozitii
Exemplu: (['Deadline-ul', 'e', 'maine', 'si', 'n-am', 'facut', 'nimic'], ['lang2', 'lang1', 'lang1', 'lang1', 'lang1', 'lang1', 'lang1'])
raw_dataset OK!


In [5]:
# ── CELL 4b — FALLBACK: upload manual daca mirror-urile nu merg ─────
# Ruleaza aceasta celula DOAR daca Cell 4 a esuat
# Descarca fisierele de pe GitHub si uploadaza-le aici

# from google.colab import files
# uploaded = files.upload()  # selectezi train.conll si dev.conll
#
# def parse_conll_file(filename):
#     with open(filename, 'r', encoding='utf-8') as f:
#         return parse_conll(f.read())
#
# raw_dataset = DatasetDict({
#     'train':      Dataset.from_dict(parse_conll_file('train.conll')),
#     'validation': Dataset.from_dict(parse_conll_file('dev.conll')),
# })
# print(f'Train: {len(raw_dataset["train"])} propozitii')
# print(f'Val:   {len(raw_dataset["validation"])} propozitii')
print('Celula fallback - decommenteaza codul daca Cell 4 a esuat')

Celula fallback - decommenteaza codul daca Cell 4 a esuat


In [5]:
# ── CELL 5 — Tokenizare ─────────────────────────────────────────────
print('Incarc tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(examples):
    tokenized = tokenizer(
        examples['words'],
        truncation=True,
        is_split_into_words=True,
    )
    all_labels = []
    for i, labels in enumerate(examples['lid']):
        word_ids = tokenized.word_ids(batch_index=i)
        aligned, prev = [], None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(labels[wid])
            else:
                aligned.append(-100)
            prev = wid
        all_labels.append(aligned)
    tokenized['labels'] = all_labels
    return tokenized

print('Tokenizez dataset-ul...')
tokenized_datasets = raw_dataset.map(
    tokenize_and_align,
    batched=True,
    remove_columns=raw_dataset['train'].column_names,
)
print('Tokenizare completa!')
print('Train tokens sample:', tokenized_datasets['train'][0]['input_ids'][:10])

Incarc tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizez dataset-ul...


Map:   0%|          | 0/2550 [00:00<?, ? examples/s]

Map:   0%|          | 0/450 [00:00<?, ? examples/s]

Tokenizare completa!
Train tokens sample: [0, 1155, 14, 444, 14110, 36, 3546, 4393, 98, 3514]


In [7]:
# CELL 6 — Antrenare
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [id2label[p] for p, l in zip(pred, label) if l != -100]
        for pred, label in zip(predictions, labels)
    ]
    return {'f1': f1_score(true_labels, true_preds)}

print('Incarc modelul XLM-RoBERTa-base...')
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,   # fix pentru UNEXPECTED/MISSING keys
)

args = TrainingArguments(
    output_dir='/content/cs_checkpoints',
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    weight_decay=0.01,
    logging_steps=20,
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    processing_class=tokenizer,     # fix pentru 'tokenizer' argument
    compute_metrics=compute_metrics,
)

print(f'Incep antrenarea... ({EPOCHS} epoci, batch {BATCH_SIZE}, GPU: {torch.cuda.is_available()})')
trainer.train()
print('Antrenare completa!')

Incarc modelul XLM-RoBERTa-base...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.bias             | MISSING    | 
classifier.weight           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Incep antrenarea... (3 epoci, batch 32, GPU: True)


Epoch,Training Loss,Validation Loss,F1
1,0.028382,0.009428,1.000000
2,0.006310,0.000426,1.000000
3,0.002406,0.000320,1.000000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: lang2 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: lang1 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: other seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Antrenare completa!


In [8]:
# ── CELL 7 — Salvare model ──────────────────────────────────────────
print(f'Salvez modelul in {SAVE_PATH}...')
os.makedirs(SAVE_PATH, exist_ok=True)
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print('Model salvat!')
print('Fisiere salvate:', os.listdir(SAVE_PATH))

Salvez modelul in /content/cs_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model salvat!
Fisiere salvate: ['tokenizer.json', 'config.json', 'tokenizer_config.json', 'model.safetensors']


In [9]:
# ── CELL 8 — Test rapid ─────────────────────────────────────────────
def predict(sentence):
    words = sentence.split()
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
    )
    word_ids = encoding.word_ids(batch_index=0)
    model.eval()
    with torch.no_grad():
        logits = model(**encoding).logits
    preds = logits.argmax(dim=-1)[0].tolist()
    seen, result = set(), []
    for wid, pid in zip(word_ids, preds):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        raw = id2label[pid]
        label = 'RO' if raw == 'lang1' else 'EN' if raw == 'lang2' else 'OTHER'
        result.append((words[wid], label))
    return result

test_sentences = [
    'Ce faci bro how are you',
    'Merg la gym dupa work',
    'Sunt asa tired dupa shift',
    'Deadline-ul e maine si n-am facut nimic',
]

print('Test predictii:')
for s in test_sentences:
    result = predict(s)
    print(f'\n"{s}"')
    for word, label in result:
        print(f'  {word:20s} -> {label}')

Test predictii:


RuntimeError: Expected all tensors to be on the same device, but got index is on cpu, different from other tensors on cuda:0 (when checking argument in method wrapper_CUDA_gather)

In [10]:
# ── CELL 9 — Download cs_model.zip ──────────────────────────────────
import shutil
from google.colab import files

print('Creez arhiva cs_model.zip...')
shutil.make_archive('/content/cs_model', 'zip', '/content/cs_model')
print('Descarcare automata...')
files.download('/content/cs_model.zip')
print('Gata! Dezarhiveaza cs_model.zip si pune folderul cs_model/ in proiect.')

Creez arhiva cs_model.zip...
Descarcare automata...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Gata! Dezarhiveaza cs_model.zip si pune folderul cs_model/ in proiect.
